# 06 - Vision Module Debug

Notebook này dùng để **quan sát ảnh và tạo profile candidate**. Phần triển khai dài được thu gọn mặc định; các cell thực thi vẫn giữ output cần thiết cho debug.

### BƯỚC 1: Setup Độc Lập

- **Tác dụng chính:** Notebook này dùng để quan sát ảnh và tạo profile candidate. Phần triển khai dài được thu gọn mặc định; các cell thực thi vẫn giữ output cần thiết cho debug.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `find_project_root`, `PROJECT_ROOT` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [1]:
import json

import sys

from pathlib import Path

from pprint import pprint

def find_project_root(start: Path | None = None) -> Path:
    """Đi ngược cây thư mục cho tới khi thấy `app/core/vision.py`.

    Args:
        start (Path | None): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        Path: Các candidate đã truy hồi, lọc hoặc xếp hạng.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "app" / "core" / "vision.py").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục Chatbot_Fashion")


In [2]:
PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.core.intent import route_user_request

from app.core.vision import (
    analyze_person_image,
    caption_product_image,
    describe_image_for_routing,
)

print(f"[OK] PROJECT_ROOT={PROJECT_ROOT}")


[OK] PROJECT_ROOT=D:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion


### BƯỚC 2: Bản Đồ Dữ Liệu Của Một Lượt Có Ảnh

- **Tác dụng chính:** Thực hiện bước quan sát ảnh và tạo profile candidate.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `inspect_image_turn` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [3]:
def inspect_image_turn(image_path: str | Path, user_query: str = "") -> dict:
    """Chạy VLM observation rồi đưa observation sang deterministic router.

    Args:
        image_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
        user_query (str): Nội dung truy vấn hoặc văn bản đầu vào.

    Returns:
        dict: Kết quả đã xử lý của hàm.
    """
    image_path = Path(image_path)
    if not image_path.exists():
        return {"status": "invalid_path", "path": str(image_path)}
    observation = describe_image_for_routing(str(image_path), user_query)
    decision = route_user_request(
        user_query,
        has_image=True,
        image_context=observation,
    )
    result = {
        "status": "ok",
        "observation": observation,
        "decision": decision.to_debug_dict(),
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result


### BƯỚC 4: Policy Sau Khi Hiểu Ảnh

- **Tác dụng chính:** Thực hiện bước quan sát ảnh và tạo profile candidate.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `MOCK_OBSERVATIONS` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [4]:
MOCK_OBSERVATIONS = [
    {"subject":"product", "caption":"một áo blazer đen", "fashion_item":"áo blazer đen", "confidence":0.92},
    {"subject":"unclear", "caption":"một người đứng trước gương", "fashion_item":"", "confidence":0.30},
]


In [5]:
for observation in MOCK_OBSERVATIONS:
    decision = route_user_request("", has_image=True, image_context=observation)
    print("\n---")
    print(json.dumps(decision.to_debug_dict(), ensure_ascii=False, indent=2))



---
{
  "intent": "product_discovery",
  "modality": "image",
  "action": "find_similar",
  "route": "image_product_search",
  "confidence": 0.92,
  "rewrite_query": "Tìm sản phẩm giống áo blazer đen trong ảnh",
  "entities": {},
  "image_context": {
    "subject": "product",
    "caption": "một áo blazer đen",
    "fashion_item": "áo blazer đen",
    "confidence": 0.92
  },
  "missing_slots": [],
  "needs_clarification": false,
  "clarification_question": "",
  "clarification_options": [],
  "follow_up_question": "Mình nhận ra một áo blazer đen. Bạn có muốn mình gợi ý cách phối đồ với món này không?",
  "follow_up_options": [
    {
      "label": "Phối đồ với món này",
      "value": "Phối đồ với món này",
      "action": "style_image_item"
    },
    {
      "label": "Không cần phối đồ",
      "value": "Không cần",
      "action": "keep_search_results"
    }
  ],
  "workflow": [
    "image_product_search"
  ],
  "reason": "VLM nhận diện được món thời trang; mặc định tìm sản phẩm tươ

### BƯỚC 5: Phân Tích Người Chỉ Tạo Candidate

- **Tác dụng chính:** Thực hiện bước quan sát ảnh và tạo profile candidate.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `preview_profile_candidate` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [6]:
def preview_profile_candidate(image_path: str | Path) -> dict:
    """Phân tích ảnh người nhưng tuyệt đối không ghi session/profile.

    Args:
        image_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.

    Returns:
        dict: Kết quả đã xử lý của hàm.
    """
    raw = analyze_person_image(str(image_path))
    candidate = {
        key: raw.get(key)
        for key in ("dang_nguoi", "tone_da")
        if raw.get(key)
    }
    output = {"candidate": candidate, "comment": raw.get("nhan_xet", ""), "saved": False}
    print(json.dumps(output, ensure_ascii=False, indent=2))
    return output


### BƯỚC 6: Smoke Test Thật (Mặc Định Không Gọi Model)

- **Tác dụng chính:** Thực hiện bước quan sát ảnh và tạo profile candidate.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `RUN_VISION_DEMO`, `TEST_IMAGE_PATH`, `TEST_USER_QUERY` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [9]:
RUN_VISION_DEMO = True

TEST_IMAGE_PATH = PROJECT_ROOT / "tests" / "sample_images" / "bodyNu.jpg"

TEST_USER_QUERY = ""


In [10]:
if RUN_VISION_DEMO:
    vision_demo = inspect_image_turn(TEST_IMAGE_PATH, TEST_USER_QUERY)
else:
    print("[SKIP] Đặt RUN_VISION_DEMO=True để gọi VLM thật.")


{
  "status": "ok",
  "observation": {
    "subject": "person",
    "caption": "Nàng gái trẻ mặc áo crop top trắng và quần jeans đứng trước cửa hàng Dior.",
    "fashion_item": "",
    "confidence": 0.9,
    "reason": "Anh ảnh tập trung vào một người phụ nữ mặc trang phục thời trang."
  },
  "decision": {
    "intent": "unknown",
    "modality": "image",
    "action": "clarify",
    "route": null,
    "confidence": 0.0,
    "rewrite_query": "",
    "entities": {},
    "image_context": {
      "subject": "person",
      "caption": "Nàng gái trẻ mặc áo crop top trắng và quần jeans đứng trước cửa hàng Dior.",
      "fashion_item": "",
      "confidence": 0.9,
      "reason": "Anh ảnh tập trung vào một người phụ nữ mặc trang phục thời trang."
    },
    "missing_slots": [
      "image_goal"
    ],
    "needs_clarification": true,
    "clarification_question": "Mình thấy Nàng gái trẻ mặc áo crop top trắng và quần jeans đứng trước cửa hàng Dior., nhưng chưa xác định chắc món bạn quan tâm. Bạ

## Kết Luận

Notebook 06 chỉ sở hữu **visual observation**. Business intent và route hữu hạn nằm ở notebook 08/`app/core/intent.py`; orchestration nằm ở notebook 09/API. Nếu VLM nhìn đúng nhưng route sai, sửa policy router. Nếu observation đã sai, sửa prompt/model vision.
